# Lecture 7 — Convolutional Networks

**Course:** UMich EECS 498-007 / 598-005 (Justin Johnson), Fall 2019, Lecture 7.  
**Merged with:** Stanford CS231n 2026 [Lecture 5: *Image Classification with CNNs*](https://cs231n.stanford.edu/slides/2026/lecture_5.pdf) and the layers/normalization part of [Lecture 6](https://cs231n.stanford.edu/slides/2026/lecture_6.pdf).

> **About the sources.** UMich-specific content is based on knowledge of Justin Johnson's lecture; the CS231n 2026 slides were read in full (many of their convolution slides say "Slide inspiration: Justin Johnson"). **[CS231n 2026]** marks content only there; **🔎** marks material beyond both.

### How to read this notebook
* Each **code cell does one thing**; explanations come before it and **"What to notice"** after important results.
* Every function says **what it computes, what goes in, and what comes out**.
* Each operation is first computed **by hand on a tiny array**, then implemented, then tested. **✍️ Try it** boxes suggest experiments. Run cells **in order**.

**The problem:** our classifiers so far **ignore the spatial structure of images**. A fully connected layer flattens a 32×32×3 image into 3,072 numbers, so it has no idea which pixels are neighbors; shift the image by one pixel and, to the network, every input changes. **The solution:** layers designed for images: **convolution** and **pooling** (plus **normalization**, which makes deep networks trainable). *Convolution and pooling extract features while respecting 2-D structure; fully connected layers at the end predict scores; everything is trained end-to-end with backprop* **[CS231n 2026]**.

**History [CS231n 2026]:** LeNet (LeCun et al. 1998) → AlexNet (2012) → **2012–2020: ConvNets dominate all vision tasks** (detection, segmentation, captioning, text-to-image) → **2021–present: Transformers have taken over** many of them (Lecture 13). Convolutions remain everywhere: in efficient models, in hybrids, and in diffusion U-Nets.

| § | Topic | What you'll build |
|---|---|---|
| 1 | Fully connected vs. convolution layers | Parameter counts |
| 2 | **The convolution layer** | A convolution by hand; loops; **im2col**; backward pass; gradient check |
| 3 | Stacking convolutions; what filters learn | Why a nonlinearity is needed |
| 4 | **Spatial dimensions**: padding, stride, **receptive fields** | Output-size formula; the slides' 760 / 768K example; receptive fields measured |
| 5 | 1×1 and other convolutions | 1×1 conv = a per-pixel fully connected layer |
| 6 | **Pooling** | The slides' example; the backward pass |
| 7 | **Translation equivariance** [CS231n 2026] | Measured |
| 8 | LeNet-5; training a small CNN | CNN vs. MLP: accuracy, parameters, shift robustness |
| 9 | **Batch normalization**; Layer/Instance/Group norm | BN step by step; train vs. test; folding; a deep net with and without BN |
| 10 | PyTorch layers | (Colab) |

### Setup

In [ ]:
import numpy as np                          # arrays and math
import matplotlib.pyplot as plt             # plots
from scipy import ndimage, signal           # image shifting; reference correlation for checking our convolution
from sklearn.datasets import load_digits    # the 8x8 handwritten digits
import math, time

np.set_printoptions(precision=4, suppress=True)
digits = load_digits()
images, labels = digits.images / 16.0, digits.target
split = np.random.default_rng(0).permutation(len(images)); tr, te = split[:1300], split[1300:]

---
## 1. Fully connected layer vs. convolution layer

**Fully connected (FC) layer:** stretch a 32×32×3 image into a 3072×1 vector, multiply by a 10×3072 weight matrix. Each output is **one number: a dot product between one row of W and the whole input** (3,072-dimensional). Every output looks at every pixel.

**Convolution layer:** keep the image as **3×32×32** (channels × height × width). Take a small **filter**, e.g. 3×5×5 (*filters always extend the full depth of the input*), and **slide it over the image, computing a dot product at every position**. Each position gives **one number: the dot product of the filter with a 3×5×5 chunk of the image** (75-dimensional, + a bias). All positions together form a 28×28 **activation map**.

Two big ideas:
* **Local connectivity:** each output looks only at a small neighborhood.
* **Weight sharing:** the same filter (the same 76 numbers) is used at every position.

The payoff in parameters:

In [ ]:
C, H, W = 3, 224, 224
print(f"FC layer, {C}x{H}x{W} image -> 4,096 units: {C * H * W * 4096:,} weights")
print(f"conv layer, 64 filters of size {C}x3x3:       {64 * (C * 3 * 3 + 1):,} weights (including biases)")

---
## 2. The convolution layer

* **6 filters** of size 3×5×5 on a 3×32×32 image give **6 activation maps** of 1×28×28; **stack them into a 6×28×28 output**. The weights form a **6×3×5×5** tensor, plus a **6-dim bias**. (Equivalently: a 28×28 grid with a 6-dim vector at each point.)
* **With batches:** input $N\times C_{\text{in}}\times H\times W$, filters $C_{\text{out}}\times C_{\text{in}}\times K_h\times K_w$ → output $N\times C_{\text{out}}\times H'\times W'$.

$$\text{out}[n, f, i, j] = b_f + \sum_{c}\sum_{u}\sum_{v}w[f, c, u, v]\cdot x_{\text{pad}}[n, c,\ iS + u,\ jS + v]$$
(🔎 Strictly this is **cross-correlation**: true convolution flips the filter first. Since filters are learned, it doesn't matter, and every framework computes this.)

### 2.1 By hand first
A 1-channel 4×4 image and a 2×2 filter, no padding, stride 1. Output size: $4 - 2 + 1 = 3$. The top-left output is the dot product of the filter with the top-left 2×2 patch:

In [ ]:
x_tiny = np.array([[1, 2, 0, 1],
                   [0, 1, 3, 2],
                   [2, 1, 0, 1],
                   [1, 0, 2, 3]], float)
w_tiny = np.array([[1, 0],
                   [0, -1]], float)                      # "top-left minus bottom-right" (the Roberts filter of Lecture 1)
patch = x_tiny[0:2, 0:2]
print("top-left patch:\n", patch)
print("dot product with the filter: 1*1 + 2*0 + 0*0 + 1*(-1) =", np.sum(patch * w_tiny))

Sliding to all 3×3 positions:

In [ ]:
out_tiny = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        out_tiny[i, j] = np.sum(x_tiny[i:i + 2, j:j + 2] * w_tiny)    # the 2x2 patch at (i, j), times the filter
print("output (3 x 3):\n", out_tiny)

### 2.2 The naive implementation
Exactly the formula: loops over images, filters, and output positions. Clear, and slow.

In [ ]:
def conv_forward_naive(x, w, b, stride=1, pad=0):
    # x: (N, C, H, W) inputs; w: (F, C, KH, KW) filters; b: (F,) biases. Returns (N, F, H', W').
    N, C, H, W = x.shape; F, _, KH, KW = w.shape
    xp = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)))           # zero-pad height and width only
    Ho, Wo = (H + 2 * pad - KH) // stride + 1, (W + 2 * pad - KW) // stride + 1
    out = np.zeros((N, F, Ho, Wo))
    for n in range(N):
        for f in range(F):
            for i in range(Ho):
                for j in range(Wo):
                    patch = xp[n, :, i * stride:i * stride + KH, j * stride:j * stride + KW]   # C x KH x KW chunk
                    out[n, f, i, j] = np.sum(patch * w[f]) + b[f]
    return out

print("naive conv on the tiny example:\n", conv_forward_naive(x_tiny[None, None], w_tiny[None, None], np.zeros(1))[0, 0])

### 2.3 im2col: convolution as one matrix multiply
Fast libraries rearrange every receptive-field patch into a **column** ("image to columns"). Then the whole convolution is a single matrix multiply: (filters as rows) × (patches as columns). For the tiny example, there are 9 patches of 4 numbers each:

In [ ]:
def im2col(x, KH, KW, stride, pad):
    # Rearrange every KH x KW patch of x (N, C, H, W) into a column.
    # Returns cols of shape (N, C*KH*KW, H'*W'), and the output size (H', W').
    N, C, H, W = x.shape
    xp = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)))
    Ho, Wo = (H + 2 * pad - KH) // stride + 1, (W + 2 * pad - KW) // stride + 1
    cols = np.zeros((N, C, KH, KW, Ho, Wo))
    for u in range(KH):                                    # loop over filter offsets only (KH*KW iterations)
        for v in range(KW):
            cols[:, :, u, v] = xp[:, :, u:u + stride * Ho:stride, v:v + stride * Wo:stride]   # all positions at once
    return cols.reshape(N, C * KH * KW, Ho * Wo), Ho, Wo

cols_tiny, _, _ = im2col(x_tiny[None, None], 2, 2, 1, 0)
print("patches as columns (4 numbers each, 9 positions):\n", cols_tiny[0])
print("filter as a row times the columns:", (w_tiny.reshape(1, -1) @ cols_tiny[0]).reshape(3, 3).ravel(), "  (= the output above)")

**The im2col forward pass.** It returns a `cache` of values the backward pass will need (Lecture 6's forward/backward pattern):

In [ ]:
def conv_forward(x, w, b, stride=1, pad=0):
    # Convolution via im2col. Returns (output (N, F, H', W'), cache for the backward pass).
    F, C, KH, KW = w.shape
    cols, Ho, Wo = im2col(x, KH, KW, stride, pad)                              # (N, C*KH*KW, H'*W')
    out = (w.reshape(F, -1) @ cols).reshape(x.shape[0], F, Ho, Wo) + b[None, :, None, None]
    return out, (x, w, cols, stride, pad, Ho, Wo)

x_r = np.random.default_rng(0).normal(size=(2, 3, 7, 7))
w_r = np.random.default_rng(1).normal(size=(4, 3, 3, 3)); b_r = np.random.default_rng(2).normal(size=4)
out_naive = conv_forward_naive(x_r, w_r, b_r, stride=2, pad=1)
out_fast, cache = conv_forward(x_r, w_r, b_r, stride=2, pad=1)
print(f"output shape {out_fast.shape};  naive == im2col: {np.allclose(out_naive, out_fast)}")
reference = signal.correlate(np.pad(x_r[0], ((0, 0), (1, 1), (1, 1))), w_r[0], mode="valid")[0, ::2, ::2] + b_r[0]
print("matches SciPy's cross-correlation:", np.allclose(reference, out_fast[0, 0]))

### 2.4 The backward pass
Given the upstream gradient $\frac{\partial L}{\partial\text{out}}$ (same shape as the output):
* $\frac{\partial L}{\partial w}$: each weight multiplied every patch, so sum (upstream × patch) over all images and positions.
* $\frac{\partial L}{\partial b}$: each bias was added at every position: sum the upstream gradient over images and positions.
* $\frac{\partial L}{\partial x}$: send the upstream gradient back through the filter into each patch, then **add** the patches back into the image (**col2im**). Overlapping patches share pixels, so their contributions **accumulate with `+=`**: Lecture 6's copy rule.

In [ ]:
def col2im(dcols, x_shape, KH, KW, stride, pad, Ho, Wo):
    # Inverse of im2col for gradients: add each column's values back into the pixels it came from.
    N, C, H, W = x_shape
    dxp = np.zeros((N, C, H + 2 * pad, W + 2 * pad))
    dcols = dcols.reshape(N, C, KH, KW, Ho, Wo)
    for u in range(KH):
        for v in range(KW):
            dxp[:, :, u:u + stride * Ho:stride, v:v + stride * Wo:stride] += dcols[:, :, u, v]   # += : shared pixels
    return dxp[:, :, pad:pad + H, pad:pad + W]                                                    # remove the padding

def conv_backward(dout, cache):
    # Gradients of the convolution. dout: (N, F, H', W'). Returns (dx, dw, db).
    x, w, cols, stride, pad, Ho, Wo = cache; F, C, KH, KW = w.shape; N = x.shape[0]
    d2 = dout.reshape(N, F, Ho * Wo)
    dw = np.einsum("nfp,nkp->fk", d2, cols).reshape(w.shape)         # sum over images n and positions p
    db = dout.sum(axis=(0, 2, 3))
    dcols = np.einsum("fk,nfp->nkp", w.reshape(F, -1), d2)           # back through the filters
    return col2im(dcols, x.shape, KH, KW, stride, pad, Ho, Wo), dw, db

**Gradient check** at a few entries of $x$, $w$, $b$ (using $L = \sum\text{out}\odot\text{dout}$, whose gradient with respect to `out` is exactly `dout`):

In [ ]:
def numerical_derivative(fun, arr, idx, h=1e-6):
    # Centered-difference derivative of the scalar fun() with respect to arr[idx].
    old = arr[idx]
    arr[idx] = old + h; plus = fun()
    arr[idx] = old - h; minus = fun()
    arr[idx] = old
    return (plus - minus) / (2 * h)

dout = np.random.default_rng(3).normal(size=out_fast.shape)
dx, dw, db = conv_backward(dout, cache)
L_conv = lambda: np.sum(conv_forward(x_r, w_r, b_r, 2, 1)[0] * dout)
errors = [abs(numerical_derivative(L_conv, x_r, (0, 1, 3, 2)) - dx[0, 1, 3, 2]),
          abs(numerical_derivative(L_conv, w_r, (2, 1, 0, 2)) - dw[2, 1, 0, 2]),
          abs(numerical_derivative(L_conv, b_r, (3,)) - db[3])]
print(f"conv backward: max |analytic - numerical| = {max(errors):.1e}")

**Speed:** naive loops vs. im2col on 16 CIFAR-sized images with 8 filters of 5×5:

In [ ]:
x_big = np.random.default_rng(4).normal(size=(16, 3, 32, 32)); w_big = np.random.default_rng(5).normal(size=(8, 3, 5, 5)); b_big = np.zeros(8)
t0 = time.time(); conv_forward_naive(x_big[:2], w_big, b_big); t_naive = (time.time() - t0) * 8      # time 2 images, scale to 16
t0 = time.time(); conv_forward(x_big, w_big, b_big); t_fast = time.time() - t0
print(f"naive ~{t_naive:.1f}s, im2col {t_fast * 1000:.0f} ms")

---
## 3. Stacking convolutions, and what filters learn

**A ConvNet is a neural network with conv layers**: e.g., 3×32×32 → conv (6 filters 5×5×3) → 6×28×28 → conv (10 filters 5×5×6) → 10×24×24 → …

**Q: What happens if we stack two convolutions directly?** **A: we get another convolution!** Both are linear, and two linear operations make one (like Lecture 5's network without activations). So **we put activations (ReLU) between them**. Verified: two 3×3 convolutions in a row equal one 5×5 convolution:

In [ ]:
x1 = np.random.default_rng(6).normal(size=(1, 1, 12, 12))
wa = np.random.default_rng(7).normal(size=(1, 1, 3, 3)); wb = np.random.default_rng(8).normal(size=(1, 1, 3, 3)); zero = np.zeros(1)
two_convs = conv_forward(conv_forward(x1, wa, zero)[0], wb, zero)[0]
w_combined = signal.convolve2d(wa[0, 0], wb[0, 0])[None, None]                 # the equivalent single 5x5 filter
one_conv = conv_forward(x1, w_combined, zero)[0]
print("conv(conv(x)) == one 5x5 conv:", np.allclose(two_convs, one_conv))

**What do conv filters learn?**
* **Linear classifier:** one template per class.
* **MLP:** a bank of whole-image templates.
* **First-layer conv filters: local image templates**, often **oriented edges and opposing colors** (AlexNet's 64 filters of 3×11×11): the same detectors as Hubel & Wiesel's simple cells (Lecture 1).
* **Deeper layers** are harder to visualize; they tend to respond to larger structures (eyes, letters) (Springenberg et al., ICLR 2015). We'll see filters learned on digits in §8.

---
## 4. Spatial dimensions

**Input 7×7, filter 3×3 → output 5×5.** In general, **input $W$, filter $K$ → output $W - K + 1$.**

**Problem: feature maps shrink with each layer!** **Solution: padding**: surround the input with zeros. With padding $P$: output $W - K + 1 + 2P$. **Common: $P = (K - 1)/2$ ("same" padding)**, which keeps the size unchanged.

**Stride:** move the filter $S$ pixels at a time (7×7 input, 3×3 filter, stride 2 → 3×3 output). The general formula:
$$\text{output} = \frac{W - K + 2P}{S} + 1$$

In [ ]:
def out_size(W_, K, P=0, S=1):
    # Output width of a convolution: (W - K + 2P) // S + 1.
    return (W_ - K + 2 * P) // S + 1

for W_, K, P, S in [(7, 3, 0, 1), (7, 3, 1, 1), (7, 3, 0, 2), (32, 5, 2, 1), (224, 7, 3, 2)]:
    actual = conv_forward(np.zeros((1, 1, W_, W_)), np.zeros((1, 1, K, K)), np.zeros(1), S, P)[0].shape[-1]
    print(f"input {W_:3d}, filter {K}, pad {P}, stride {S}: formula {out_size(W_, K, P, S):3d}, our conv gives {actual:3d}")

### 4.1 The slides' example
**Input 3×32×32; 10 filters of 5×5, stride 1, pad 2.** Output size, number of parameters, number of multiply-adds:

In [ ]:
C_in, H_in, F, K, S, P = 3, 32, 10, 5, 1, 2
H_out = out_size(H_in, K, P, S)
print(f"output volume: {F} x {H_out} x {H_out}        ((32 + 2*2 - 5) / 1 + 1 = 32)")
print(f"parameters: {F} filters x ({C_in}*{K}*{K} + 1) = {F * (C_in * K * K + 1)}      (76 per filter)")
print(f"multiply-adds: {F * H_out * H_out:,} outputs x {C_in * K * K} each = {F * H_out * H_out * C_in * K * K:,}")

### 4.2 Receptive fields
**With kernel size K, each output depends on a K×K region of the input** (its **receptive field**). **Each additional convolution adds K − 1, so L layers see $1 + L(K - 1)$** pixels across. *Careful: "receptive field in the input" vs. "in the previous layer".* **Problem:** for big images, it takes many layers before each output "sees" the whole image. **Solution: downsample inside the network** (stride or pooling); after a stride-2 layer, every later layer's step covers 2 input pixels, so the receptive field grows faster:
$$r_\ell = r_{\ell-1} + (K_\ell - 1)\times(\text{product of the strides of all earlier layers})$$

🔎 **We can measure a receptive field with backprop:** set the gradient to 1 at a single output pixel, backpropagate to the input, and see which input pixels get a nonzero gradient. (Using all-ones filters so no gradient cancels by accident.)

In [ ]:
def measured_receptive_field(layers, size=41):
    # layers: list of (K, stride). Returns the width of the input region that affects the central output pixel.
    x = np.zeros((1, 1, size, size)); caches = []
    for K, S in layers:                                                   # forward through the stack
        x, c = conv_forward(x, np.ones((1, 1, K, K)), np.zeros(1), S, 0); caches.append(c)
    grad = np.zeros_like(x); grad[0, 0, x.shape[2] // 2, x.shape[3] // 2] = 1.0   # gradient 1 at ONE output pixel
    for c in reversed(caches):                                            # backward to the input
        grad = conv_backward(grad, c)[0]
    rows = np.nonzero(np.abs(grad[0, 0]).sum(axis=1))[0]
    return rows.max() - rows.min() + 1

def formula_receptive_field(layers):
    # The formula above.
    rf, jump = 1, 1
    for K, S in layers:
        rf += (K - 1) * jump; jump *= S
    return rf

for layers in [[(3, 1)], [(3, 1)] * 3, [(3, 1)] * 5, [(5, 1)] * 2, [(3, 1), (3, 2), (3, 1), (3, 2), (3, 1)]]:
    print(f"(K, stride) = {str(layers):45s}: measured {measured_receptive_field(layers):2d}, formula {formula_receptive_field(layers):2d}")

**What to notice:** five 3×3 layers see 11 pixels across (1 + 5·2). Making two of the five layers stride 2 grows the same depth's receptive field to 21 (1 + 2 + 2 + 4 + 4 + 8), because after each stride-2 layer, every step covers twice as many input pixels.

### 4.3 Convolution summary
* **Input:** $C_{\text{in}}\times H\times W$. **Hyperparameters:** kernel size $K_H\times K_W$, number of filters $C_{\text{out}}$, padding $P$, stride $S$.
* **Weights:** $C_{\text{out}}\times C_{\text{in}}\times K_H\times K_W$ (i.e., $C_{\text{out}}$ filters of size $C_{\text{in}}\times K_H\times K_W$), plus a bias of $C_{\text{out}}$.
* **Output:** $C_{\text{out}}\times H'\times W'$, with $H' = (H - K + 2P)/S + 1$.
* **Common settings:** square filters; "same" padding $P = (K - 1)/2$; channels 32, 64, 128, 256; **K = 3, P = 1, S = 1** (3×3 conv); **K = 5, P = 2, S = 1**; **K = 1, P = 0, S = 1** (1×1 conv); **K = 3, P = 1, S = 2** (downsample by 2).

---
## 5. Other kinds of convolution

**1×1 convolution:** e.g., 64×56×56 input, 32 filters of size 1×1×64 → 32×56×56. Each output is a dot product **across channels at a single pixel**, so a stack of 1×1 convolutions is **an MLP applied at every pixel** (*Network in Network*, Lin et al. 2014). Used to change the number of channels cheaply (Lecture 8). Checked:

In [ ]:
x56 = np.random.default_rng(9).normal(size=(2, 64, 6, 6)); w1x1 = np.random.default_rng(10).normal(size=(32, 64, 1, 1)); b1x1 = np.zeros(32)
conv_result = conv_forward(x56, w1x1, b1x1)[0]
fc_at_each_pixel = np.einsum("nchw,fc->nfhw", x56, w1x1[:, :, 0, 0])            # the same FC layer at every (h, w)
print("1x1 convolution == a fully connected layer applied at every pixel:", np.allclose(conv_result, fc_at_each_pixel))

* **1-D convolution:** input $C_{\text{in}}\times W$, weights $C_{\text{out}}\times C_{\text{in}}\times K$ (audio, text, time series).
* **3-D convolution:** input $C_{\text{in}}\times H\times W\times D$, weights $C_{\text{out}}\times C_{\text{in}}\times K\times K\times K$ (video, medical volumes; Lecture 18).
* PyTorch: `nn.Conv1d`, `nn.Conv2d`, `nn.Conv3d`.

🔎 **Two options the slides skip** (*"We didn't talk about groups or dilation…"* **[CS231n 2026]**):
* **Dilation** $d$ spaces the filter taps apart: a 3×3 filter with dilation 2 covers 5×5 using only 9 weights (effective size $d(K - 1) + 1$). Stacked, receptive fields grow **exponentially** with depth (segmentation, audio).
* **Groups** $G$ split the channels into $G$ independent groups, each with its own filters, cutting parameters and compute by $G$. $G = C_{\text{in}}$ is a **depthwise** convolution (MobileNet, Lecture 8).

---
## 6. Pooling layers: another way to downsample

For an input $C\times H\times W$, **pooling downsamples each $1\times H\times W$ channel separately** (e.g., 64×224×224 → 64×112×112). **Hyperparameters:** kernel size, stride, and the pooling function (max or average). **No learnable parameters.**

**The slides' example by hand:** max pooling, 2×2 kernel, stride 2, on a 4×4 input: split it into four 2×2 blocks and keep each block's maximum:

In [ ]:
x_pool = np.array([[1, 1, 2, 4],
                   [5, 6, 7, 8],
                   [3, 2, 1, 0],
                   [1, 2, 3, 4]], float)
for (r0, c0) in [(0, 0), (0, 2), (2, 0), (2, 2)]:
    block = x_pool[r0:r0 + 2, c0:c0 + 2]
    print(f"block at ({r0},{c0}): {block.ravel().tolist()} -> max {block.max():g}, average {block.mean():g}")

**As a function, with its backward pass.** Max pooling's backward is Lecture 6's **max node**: the gradient goes only to the input that won (split evenly if there's a tie). Average pooling spreads it equally over the window.

In [ ]:
def pool_forward(x, K=2, S=2, mode="max"):
    # Max or average pooling of x (N, C, H, W). Returns (output, cache).
    N, C, H, W = x.shape; Ho, Wo = (H - K) // S + 1, (W - K) // S + 1
    windows = np.stack([x[:, :, u:u + S * Ho:S, v:v + S * Wo:S] for u in range(K) for v in range(K)], axis=-1)  # (N,C,Ho,Wo,K*K)
    out = windows.max(-1) if mode == "max" else windows.mean(-1)
    return out, (x.shape, windows, K, S, mode)

def pool_backward(dout, cache):
    # Gradient of pooling: max routes to the winner(s); average spreads evenly.
    shape, windows, K, S, mode = cache; dx = np.zeros(shape); Ho, Wo = dout.shape[2:]
    if mode == "max":
        share = windows == windows.max(-1, keepdims=True)
        share = share / share.sum(-1, keepdims=True)                       # split ties evenly
    else:
        share = np.full(windows.shape, 1 / (K * K))
    for k, (u, v) in enumerate([(u, v) for u in range(K) for v in range(K)]):
        dx[:, :, u:u + S * Ho:S, v:v + S * Wo:S] += share[..., k] * dout
    return dx

mx, cache_max = pool_forward(x_pool[None, None], mode="max")
av, _ = pool_forward(x_pool[None, None], mode="avg")
print("max pooling:\n", mx[0, 0], "   (slides: [[6, 8], [3, 4]])")
print("average pooling:\n", av[0, 0], "   (slides: [[3.25, 5.25], [2, 2]])")
print("max-pool backward of an all-ones upstream gradient (only the winners get it):\n", pool_backward(np.ones_like(mx), cache_max)[0, 0])

Max pooling **gives invariance to small spatial shifts** (a feature moving within its window gives the same max). **Pooling summary:** output $H' = (H - K)/S + 1$; no learnable parameters; common: **max, K = 2, S = 2** (2× downsampling); max, K = 3, S = 2 (AlexNet).

---
## 7. Translation equivariance [CS231n 2026]

**Translate, then convolve = convolve, then translate:** $\text{Conv}(\text{Translate}(X)) = \text{Translate}(\text{Conv}(X))$. **Intuition: image features don't depend on where they are in the image.** The same filter is applied everywhere, so a cat's ear produces the same feature wherever it is; it just shows up at a different place in the activation map. (Exact for whole-pixel shifts away from the borders; for pooling and strided layers, only for shifts that are multiples of the stride.)

**Test:** an object in the middle of a 20×20 image; shift it and compare.

In [ ]:
img = np.zeros((1, 1, 20, 20)); img[0, 0, 5:11, 6:9] = np.random.default_rng(11).random((6, 3))   # away from the borders
wq = np.random.default_rng(12).normal(size=(4, 1, 3, 3)); bq = np.zeros(4)

shift = lambda a, dy, dx: np.roll(np.roll(a, dy, axis=2), dx, axis=3)      # move the content by (dy, dx) pixels
conv = lambda a: conv_forward(a, wq, bq, 1, 1)[0]
pool = lambda a: pool_forward(a, 2, 2)[0]

print("conv(shift(x)) == shift(conv(x)), shift (3, 4):", np.allclose(conv(shift(img, 3, 4)), shift(conv(img), 3, 4)))
print("pool(shift by 2) == shift(pool) by 1:", np.allclose(pool(shift(img, 2, 2)), shift(pool(img), 1, 1)), "(2 = the stride)")
print("pool(shift by 1) == pool unchanged:", np.allclose(pool(shift(img, 1, 1)), pool(img)), "(not exact for shifts smaller than the stride)")

🔎 That last line matters in practice: strided layers make CNNs only **approximately** shift-invariant. Small shifts can change a modern CNN's prediction (Zhang, ICML 2019, *Making Convolutional Networks Shift-Invariant Again*, which fixed it with blurring before downsampling).

---
## 8. A classic architecture, and training a small CNN

**Classic pattern:** $[\text{Conv, ReLU, Pool}]\times N$, flatten, $[\text{FC, ReLU}]\times N$, FC. **Example: LeNet-5** (LeCun et al. 1998), in the UMich lecture's form. Shapes and parameters, computed layer by layer:

In [ ]:
def lenet_table():
    # Walk through LeNet-5, printing each layer's output shape and parameter count.
    C, Hh, total = 1, 28, 0
    print(f"{'layer':32s} {'output':>14s} {'params':>10s}")
    print(f"{'input':32s} {str((1, 28, 28)):>14s}")
    for name, kind, args in [("conv (C_out=20, K=5, P=2, S=1)", "conv", (20, 5, 2, 1)), ("ReLU", "act", None), ("max pool (K=2, S=2)", "pool", (2, 2)),
                             ("conv (C_out=50, K=5, P=2, S=1)", "conv", (50, 5, 2, 1)), ("ReLU", "act", None), ("max pool (K=2, S=2)", "pool", (2, 2)),
                             ("flatten", "flat", None), ("linear (2450 -> 500)", "fc", 500), ("ReLU", "act", None), ("linear (500 -> 10)", "fc", 10)]:
        p = 0
        if kind == "conv":
            F_, K_, P_, S_ = args; p = F_ * (C * K_ * K_ + 1); C, Hh = F_, out_size(Hh, K_, P_, S_); shape = (C, Hh, Hh)
        elif kind == "pool":
            Hh = out_size(Hh, args[0], 0, args[1]); shape = (C, Hh, Hh)
        elif kind == "flat":
            C = C * Hh * Hh; shape = (C,)
        elif kind == "fc":
            p = args * (C + 1); C = args; shape = (C,)
        else:
            shape = (C, Hh, Hh) if Hh > 1 else (C,)
        total += p
        print(f"{name:32s} {str(shape):>14s} {p:>10,d}")
    print(f"total parameters: {total:,}")
lenet_table()

**What to notice:** **as we go deeper, the spatial size decreases** (via pooling or strided conv) **and the number of channels increases**: the network trades "where" for "what". Most parameters are in the first FC layer.

**Now train a small CNN on our 8×8 digits**, built from our own layers: conv (8 filters, 3×3, pad 1) → ReLU → max pool 2×2 → FC → 10 scores. We'll compare it with a 2-layer MLP on normal digits and on digits **shifted by one pixel**.

**Step 1: the softmax loss and the CNN** (forward and backward, each layer in turn):

In [ ]:
def softmax_xent(S_, y_):
    # Softmax loss and its gradient with respect to the scores.
    S_ = S_ - S_.max(1, keepdims=True); P_ = np.exp(S_); P_ /= P_.sum(1, keepdims=True)
    loss = -np.log(P_[np.arange(len(y_)), y_]).mean(); d = P_.copy(); d[np.arange(len(y_)), y_] -= 1
    return loss, d / len(y_)

class SmallCNN:
    # conv(F filters, 3x3, pad 1) -> ReLU -> max pool 2x2 -> fully connected -> 10 scores. Input: (N, 1, 8, 8).
    def __init__(self, F=8, seed=0):
        r = np.random.default_rng(seed)
        self.p = {"w": r.normal(0, math.sqrt(2 / 9), (F, 1, 3, 3)), "b": np.zeros(F),
                  "W2": r.normal(0, math.sqrt(2 / (F * 16)), (F * 16, 10)), "b2": np.zeros(10)}

    def forward(self, X, y=None):
        # Returns scores; if y is given, returns (loss, gradients).
        c1, conv_cache = conv_forward(X, self.p["w"], self.p["b"], 1, 1)     # (N, F, 8, 8)
        r1 = np.maximum(c1, 0)                                               # ReLU
        p1, pool_cache = pool_forward(r1, 2, 2)                              # (N, F, 4, 4)
        flat = p1.reshape(len(X), -1)                                        # (N, F*16)
        S_ = flat @ self.p["W2"] + self.p["b2"]
        if y is None:
            return S_
        loss, dS = softmax_xent(S_, y)                                       # backward, layer by layer:
        g = {"W2": flat.T @ dS, "b2": dS.sum(0)}                             #   fully connected
        dp1 = (dS @ self.p["W2"].T).reshape(p1.shape)                        #   un-flatten
        dr1 = pool_backward(dp1, pool_cache)                                 #   max pool
        dc1 = dr1 * (c1 > 0)                                                 #   ReLU
        _, g["w"], g["b"] = conv_backward(dc1, conv_cache)                   #   convolution
        return loss, g

**Step 2: the MLP baseline** (64 → 100 → 10) and a shared training loop (SGD with momentum, Lecture 4):

In [ ]:
class MLP:
    # 64 -> H -> 10 fully connected network with ReLU.
    def __init__(self, H=100, seed=0):
        r = np.random.default_rng(seed)
        self.p = {"W1": r.normal(0, math.sqrt(2 / 64), (64, H)), "b1": np.zeros(H), "W2": r.normal(0, math.sqrt(2 / H), (H, 10)), "b2": np.zeros(10)}
    def forward(self, X, y=None):
        Xf = X.reshape(len(X), -1); Z = Xf @ self.p["W1"] + self.p["b1"]; Hh = np.maximum(Z, 0)
        S_ = Hh @ self.p["W2"] + self.p["b2"]
        if y is None:
            return S_
        loss, dS = softmax_xent(S_, y); dZ = (dS @ self.p["W2"].T) * (Z > 0)
        return loss, {"W2": Hh.T @ dS, "b2": dS.sum(0), "W1": Xf.T @ dZ, "b1": dZ.sum(0)}

def train(model, X, y, lr=0.02, epochs=30, batch=32, seed=0):
    # SGD with momentum 0.9 over shuffled minibatches.
    r = np.random.default_rng(seed); velocity = {k: np.zeros_like(v) for k, v in model.p.items()}
    for epoch in range(epochs):
        for idx in np.array_split(r.permutation(len(X)), len(X) // batch):
            _, g = model.forward(X[idx], y[idx])
            for k in model.p:
                velocity[k] = 0.9 * velocity[k] - lr * g[k]; model.p[k] += velocity[k]
    return model

**Step 3: data**, plus a test set shifted right by one pixel. And a gradient check of the CNN:

In [ ]:
X_tr4 = (images[tr] - 0.3)[:, None]                       # (N, 1, 8, 8), roughly centered
X_te4 = (images[te] - 0.3)[:, None]
X_te_shift = np.array([ndimage.shift(im, (0, 1), order=0, cval=0) for im in images[te]])[:, None] - 0.3

cnn_check = SmallCNN(seed=3); _, g = cnn_check.forward(X_tr4[:10], labels[tr][:10]); worst = 0
for k in cnn_check.p:
    ix = tuple(np.random.default_rng(13).integers(0, s) for s in cnn_check.p[k].shape)
    num = numerical_derivative(lambda: cnn_check.forward(X_tr4[:10], labels[tr][:10])[0], cnn_check.p[k], ix)
    worst = max(worst, abs(num - g[k][ix]) / max(abs(num), abs(g[k][ix]), 1e-10))
print(f"small CNN gradient check: max relative error {worst:.1e}")

**Step 4: train both and compare:**

In [ ]:
t0 = time.time()
cnn = train(SmallCNN(), X_tr4, labels[tr]); mlp = train(MLP(), X_tr4, labels[tr])
accuracy = lambda m, X, y: np.mean(np.argmax(m.forward(X), 1) == y)
n_params = lambda m: sum(v.size for v in m.p.values())
for name, m in [("small CNN", cnn), ("MLP (64-100-10)", mlp)]:
    print(f"{name:16s}: {n_params(m):5d} parameters;  test {accuracy(m, X_te4, labels[te]):.1%};  shifted 1 pixel {accuracy(m, X_te_shift, labels[te]):.1%}")
print(f"({time.time() - t0:.0f}s)")

In [ ]:
fig, ax = plt.subplots(1, 8, figsize=(12, 1.8))
for i in range(8):
    ax[i].imshow(cnn.p["w"][i, 0], cmap="RdBu_r"); ax[i].axis("off")
plt.suptitle("the CNN's 8 learned 3x3 filters: small local templates (edges, strokes, corners)", fontsize=9); plt.show()

**What to notice:** the CNN uses about 5× fewer parameters than the MLP, reaches nearly the same accuracy on normal digits (slightly lower here), and does clearly better on shifted digits: the value of **weight sharing plus pooling** (the shifted digit triggers the same filters one pixel over, and pooling absorbs part of the shift). Both still lose accuracy, because one pixel is 12% of an 8×8 image and one pooling layer only tolerates shifts within its 2×2 window.

---
## 9. Normalization

**Problem: deep networks are very hard to train!** (§9.3 shows it.)

### 9.1 Batch normalization (Ioffe & Szegedy, ICML 2015)
**Idea: normalize a layer's outputs to zero mean and unit variance**, computed **over the batch**, separately for each feature (channel). The original motivation was reducing "internal covariate shift"; later work suggests the main benefit is a smoother loss landscape (Santurkar et al. 2018). Either way, it **improves optimization**, and since normalization is **differentiable**, we can backprop through it.

For an input $x$ of shape $N\times D$ (N examples, D features):
$$\mu_j = \frac1N\sum_ix_{i,j},\qquad\sigma_j^2 = \frac1N\sum_i(x_{i,j} - \mu_j)^2,\qquad\hat x_{i,j} = \frac{x_{i,j} - \mu_j}{\sqrt{\sigma_j^2 + \varepsilon}}$$
**But zero mean and unit variance might be too strict**, so add a **learnable scale and shift** $\gamma_j, \beta_j$: $\ y_{i,j} = \gamma_j\hat x_{i,j} + \beta_j$. (Learning $\gamma = \sigma$, $\beta = \mu$ recovers the identity.)

**By hand first:** 4 examples, one feature, values [1, 2, 3, 6]: mean 3, variance (4 + 1 + 0 + 9)/4 = 3.5:

In [ ]:
feature = np.array([1.0, 2.0, 3.0, 6.0])
mu, var = feature.mean(), feature.var()
normalized = (feature - mu) / np.sqrt(var + 1e-5)
print(f"mean {mu}, variance {var}")
print("normalized:", normalized.round(4), "  -> mean", round(normalized.mean(), 6) + 0.0, ", std", round(normalized.std(), 4))

**At test time** there may be only one image, and predictions shouldn't depend on which other images are in the batch. So during training we also keep **running averages** of $\mu$ and $\sigma^2$, and **at test time we use those fixed values**: batch norm becomes a fixed linear function.

**For conv nets,** normalize over the batch **and** both spatial dimensions: for input $N\times C\times H\times W$, the statistics have shape $1\times C\times1\times1$ (one mean and variance per channel).

**The backward pass**, derived from the computational graph and simplified:
$$\frac{\partial L}{\partial x_i} = \frac{\gamma}{N\sqrt{\sigma^2 + \varepsilon}}\left(N\frac{\partial L}{\partial y_i} - \sum_k\frac{\partial L}{\partial y_k} - \hat x_i\sum_k\frac{\partial L}{\partial y_k}\hat x_k\right),\qquad\frac{\partial L}{\partial\gamma} = \sum_i\frac{\partial L}{\partial y_i}\hat x_i,\qquad\frac{\partial L}{\partial\beta} = \sum_i\frac{\partial L}{\partial y_i}$$

In [ ]:
class BatchNorm:
    # Batch normalization over the first axis of (N, D) inputs, with learnable gamma/beta and running statistics.
    def __init__(self, D, momentum=0.9, eps=1e-5):
        self.gamma, self.beta = np.ones(D), np.zeros(D)
        self.running_mean, self.running_var = np.zeros(D), np.ones(D)
        self.momentum, self.eps, self.training = momentum, eps, True

    def forward(self, x):
        if self.training:                                    # use this batch's statistics, and update the running ones
            mu, var = x.mean(0), x.var(0)
            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * mu
            self.running_var = self.momentum * self.running_var + (1 - self.momentum) * var
        else:                                                # test time: fixed running statistics
            mu, var = self.running_mean, self.running_var
        self.std = np.sqrt(var + self.eps)
        self.xhat = (x - mu) / self.std
        return self.gamma * self.xhat + self.beta

    def backward(self, dy):
        # Training-mode backward pass (the formula above). Stores dgamma, dbeta; returns dx.
        N = dy.shape[0]
        self.dgamma, self.dbeta = (dy * self.xhat).sum(0), dy.sum(0)
        return self.gamma / (N * self.std) * (N * dy - dy.sum(0) - self.xhat * (dy * self.xhat).sum(0))

**Check it:** outputs normalized; backward pass matches numerical derivatives.

In [ ]:
bn = BatchNorm(5); xb = np.random.default_rng(14).normal(3.0, 2.0, (8, 5))
print("training-mode output: per-feature mean", bn.forward(xb).mean(0).round(6) + 0.0, " std", bn.forward(xb).std(0).round(4))
bn.gamma = np.random.default_rng(15).normal(size=5); bn.beta = np.random.default_rng(16).normal(size=5)
out = bn.forward(xb); dy = np.random.default_rng(17).normal(size=out.shape); dxb = bn.backward(dy)
f_bn = lambda: np.sum(BatchNorm.forward(bn, xb) * dy)
worst = max(abs(numerical_derivative(f_bn, xb, (i, j)) - dxb[i, j]) for i in range(8) for j in range(5))
print(f"BatchNorm backward: max |analytic - numerical| = {worst:.1e}")

**Folding BN into the previous layer at test time.** With fixed $\mu, \sigma$:
$$\text{BN}(Wx + b) = \gamma\frac{Wx + b - \mu}{\sigma} + \beta = \underbrace{\left(\tfrac{\gamma}{\sigma}W\right)}_{W'}x + \underbrace{\tfrac{\gamma}{\sigma}(b - \mu) + \beta}_{b'}$$
one linear layer, so BN costs **nothing at inference**:

In [ ]:
Wl, bl = np.random.default_rng(18).normal(size=(5, 4)), np.random.default_rng(19).normal(size=4)
bn_fold = BatchNorm(4)
for _ in range(50):
    bn_fold.forward(np.random.default_rng(20 + _).normal(size=(32, 5)) @ Wl + bl)     # collect running statistics
bn_fold.training = False
bn_fold.gamma, bn_fold.beta = np.random.default_rng(99).normal(size=4), np.random.default_rng(98).normal(size=4)
sd = np.sqrt(bn_fold.running_var + bn_fold.eps)
W_fused = Wl * (bn_fold.gamma / sd)                                   # scale each output column
b_fused = (bl - bn_fold.running_mean) * bn_fold.gamma / sd + bn_fold.beta
xq = np.random.default_rng(21).normal(size=(6, 5))
print("test-time BN(linear(x)) == one fused linear layer:", np.allclose(bn_fold.forward(xq @ Wl + bl), xq @ W_fused + b_fused))

**Usually inserted after FC or conv layers, before the nonlinearity.** **Pros:** much easier to train deep networks; higher learning rates and faster convergence; less sensitive to initialization; acts as a regularizer; free at test time. **Cons:** not well understood theoretically; **behaves differently in training and testing: a very common source of bugs!**

**The classic bug: forgetting to switch BN to test mode.** In training mode, a batch of **one** example is normalized by its own statistics: $(x - x)/\sigma = 0$, whatever the input. And in general, each prediction depends on the other examples in the batch:

In [ ]:
bn_bug = BatchNorm(4)
for k in range(100):
    bn_bug.forward(np.random.default_rng(100 + k).normal(2.0, 3.0, (64, 4)))           # "trained" on data with mean 2, std 3
x_one = np.array([[5.0, -1.0, 2.0, 8.0]])
print("training mode, batch of 1:", bn_bug.forward(x_one).round(3), " <- all zeros, whatever the input!")
bn_bug.training = False
print("test mode (running stats):", bn_bug.forward(x_one).round(3))
bn_bug.training = True
batch_a = np.vstack([x_one, np.random.default_rng(200).normal(2, 3, (7, 4))])
batch_b = np.vstack([x_one, np.random.default_rng(201).normal(10, 1, (7, 4))])
print("training mode, the same example in two batches:", bn_bug.forward(batch_a)[0].round(2), "vs", bn_bug.forward(batch_b)[0].round(2))

In PyTorch: `model.eval()` before evaluating, `model.train()` before training.

### 9.2 Layer, instance, and group normalization
All compute $y = \gamma\frac{x - \mu}{\sigma} + \beta$; they differ in **which axes** the mean and variance are taken over. For a conv activation of shape $N\times C\times H\times W$:

| Method | Statistics over | Shape of $\mu, \sigma$ | Typical use |
|---|---|---|---|
| **Batch norm** | N, H, W | $1\times C\times1\times1$ | CNNs with decent batch sizes |
| **Layer norm** **[CS231n 2026]** | C, H, W (each sample on its own) | $N\times1\times1\times1$ | **RNNs, Transformers**; same at train and test |
| **Instance norm** | H, W (each sample and channel) | $N\times C\times1\times1$ | Style transfer |
| **Group norm** (Wu & He 2018) | groups of channels, H, W | $N\times G\times1\times1$ | Detection/segmentation (small batches) |

One function covers them all; check that each normalizes over its own axes:

In [ ]:
def normalize(x, axes, eps=1e-5):
    # Subtract the mean and divide by the std computed over `axes` (no gamma/beta).
    return (x - x.mean(axes, keepdims=True)) / np.sqrt(x.var(axes, keepdims=True) + eps)

xc = np.random.default_rng(22).normal(2.0, 3.0, (4, 6, 5, 5))       # N=4, C=6, H=W=5
G_ = 3
variants = {
    "batch norm    (N, H, W)": (normalize(xc, (0, 2, 3)), lambda y: y[:, 0]),                  # check: one channel across the batch
    "layer norm    (C, H, W)": (normalize(xc, (1, 2, 3)), lambda y: y[0]),                     # check: one sample
    "instance norm (H, W)":    (normalize(xc, (2, 3)), lambda y: y[0, 0]),                     # check: one sample, one channel
    "group norm    (3 groups)": (normalize(xc.reshape(4, G_, 2, 5, 5), (2, 3, 4)).reshape(xc.shape), lambda y: y[0, :2]),  # one group
}
for name, (y_, pick) in variants.items():
    part = pick(y_)
    print(f"{name}: over its own axes, mean {round(part.mean(), 3) + 0.0:.3f}, std {part.std():.3f}")

### 9.3 Does batch norm really make deep networks easier to train?
A **10-layer** fully connected network on the digits, with a too-small initialization (std 0.01, a common mistake). Without normalization the signal shrinks at every layer and vanishes; with BN after each linear layer, every layer's output is re-normalized, so the initialization scale stops mattering. We also try the network **without BN but with a proper (He) initialization**.

**The model** (with an optional BN layer after each hidden linear layer):

In [ ]:
class DeepMLP:
    # depth linear layers of width `width`, ReLU between them, optional BatchNorm before each ReLU.
    def __init__(self, depth=10, width=64, init_std=0.01, use_bn=False, seed=0):
        r = np.random.default_rng(seed); sizes = [64] + [width] * (depth - 1) + [10]
        self.W = [(init_std * r.normal(size=(a, b_)) if init_std else r.normal(0, math.sqrt(2 / a), (a, b_)))   # None = He init
                  for a, b_ in zip(sizes[:-1], sizes[1:])]
        self.b = [np.zeros(b_) for b_ in sizes[1:]]
        self.bn = [BatchNorm(b_) if use_bn else None for b_ in sizes[1:-1]]

    def step(self, X, y, lr):
        # One forward + backward + SGD update. Returns the loss.
        acts, inputs, pre = X, [], {}
        for i in range(len(self.W)):
            inputs.append(acts)
            z = acts @ self.W[i] + self.b[i]
            if i < len(self.W) - 1:                                  # hidden layer: (BN) -> ReLU
                if self.bn[i]: z = self.bn[i].forward(z)
                pre[i] = z; acts = np.maximum(z, 0)
            else:
                acts = z                                             # output scores
        loss, d = softmax_xent(acts, y)
        for i in reversed(range(len(self.W))):
            if i < len(self.W) - 1:
                d = d * (pre[i] > 0)                                 # through ReLU
                if self.bn[i]:
                    d = self.bn[i].backward(d)                       # through BN
                    self.bn[i].gamma -= lr * self.bn[i].dgamma; self.bn[i].beta -= lr * self.bn[i].dbeta
            gW, gb = inputs[i].T @ d, d.sum(0)
            d = d @ self.W[i].T                                      # through the linear layer
            self.W[i] -= lr * gW; self.b[i] -= lr * gb
        return loss

**Train each version for 600 steps:**

In [ ]:
X_flat = images[tr].reshape(len(tr), -1) - 0.3
for name, kw in [("10 layers, init std 0.01, no BN", dict(init_std=0.01, use_bn=False)),
                 ("10 layers, init std 0.01, with BN", dict(init_std=0.01, use_bn=True)),
                 ("10 layers, He init, no BN", dict(init_std=None, use_bn=False))]:
    model, r, history = DeepMLP(**kw), np.random.default_rng(0), []
    for t in range(600):
        idx = r.integers(0, len(X_flat), 64)
        history.append(model.step(X_flat[idx], labels[tr][idx], 0.05))
    print(f"{name:36s}: training loss, first 50 steps {np.mean(history[:50]):.3f} -> last 50 steps {np.mean(history[-50:]):.3f}")

**What to notice:** without BN, the badly initialized network never leaves the chance-level loss (log 10 ≈ 2.30). The same network with BN trains normally, and so does the network without BN but with a properly scaled initialization (the other half of the story, Lecture 10). BN makes training robust to choices like the initialization scale, a big part of why it became standard.

---
## 10. PyTorch layers (run in Colab)

In [ ]:
# [PyTorch] Needs: pip install torch.
import torch, torch.nn as nn
conv = nn.Conv2d(in_channels=3, out_channels=10, kernel_size=5, stride=1, padding=2)
print("Conv2d weight", tuple(conv.weight.shape), "bias", tuple(conv.bias.shape), "-> params", sum(p.numel() for p in conv.parameters()))
print("output shape", tuple(conv(torch.randn(2, 3, 32, 32)).shape))
lenet = nn.Sequential(nn.Conv2d(1, 20, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2, 2),
                      nn.Conv2d(20, 50, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2, 2),
                      nn.Flatten(), nn.Linear(2450, 500), nn.ReLU(), nn.Linear(500, 10))
print("LeNet-5 parameters:", sum(p.numel() for p in lenet.parameters()), "; output", tuple(lenet(torch.randn(4, 1, 28, 28)).shape))
bn = nn.BatchNorm2d(10)
bn.train(); _ = bn(torch.randn(8, 10, 4, 4))     # training mode: batch statistics, updates running stats
bn.eval()                                        # test mode: running statistics. Forgetting this is the classic bug!
print("also: nn.Conv1d, nn.Conv3d, nn.AvgPool2d, nn.LayerNorm, nn.GroupNorm(groups, C), nn.InstanceNorm2d, Conv2d(..., dilation=2, groups=G)")

---
## 11. Summary

**Components of a CNN:** convolution layers, pooling layers, fully connected layers, activation functions, normalization (sometimes dropout). **The most computationally expensive part is the convolution.**

| Topic | Takeaway |
|---|---|
| Convolution | Slide a $C_{\text{in}}\times K\times K$ filter over the image, computing dot products; local connections, shared weights |
| Shapes | $N\times C_{\text{in}}\times H\times W\to N\times C_{\text{out}}\times H'\times W'$, $H' = (H - K + 2P)/S + 1$; weights $C_{\text{out}}\times C_{\text{in}}\times K\times K$ |
| im2col | Convolution becomes one matrix multiply; backward accumulates (+=) over shared pixels |
| Stacking | Conv∘conv is still a conv; put ReLUs between |
| Receptive field | $1 + L(K - 1)$ at stride 1; downsampling grows it faster |
| 1×1 conv | An MLP at every pixel |
| Pooling | Max/average downsampling, no parameters; max routes the gradient |
| Equivariance | Conv(shift(x)) = shift(Conv(x)); pooling/stride only for multiples of the stride |
| LeNet pattern | [Conv, ReLU, Pool]×N → FC; spatial size ↓, channels ↑ |
| BatchNorm | Normalize each channel over the batch, then γx̂ + β; running stats at test time; foldable; train/test bugs |
| Other norms | Layer (Transformers), instance (style), group (small batches) |

### Check yourself
<details><summary>1. Input 3×32×32, 10 filters of 5×5, stride 1, pad 2: output shape, parameters, multiply-adds?</summary>

10×32×32; 10 × (3·5·5 + 1) = 760 parameters; 10·32·32 = 10,240 outputs × 75 = 768,000 multiply-adds.
</details>

<details><summary>2. What's the receptive field of three stacked 3×3 stride-1 convolutions?</summary>

1 + 3·2 = 7, the same as one 7×7 convolution (VGG's argument, Lecture 8).
</details>

<details><summary>3. Why must there be a nonlinearity between two convolutions?</summary>

Convolutions are linear; two in a row equal one convolution with a bigger kernel, so the extra layer adds no expressive power.
</details>

<details><summary>4. Why does batch norm behave differently during training and testing?</summary>

During training it uses the current batch's statistics; at test time it uses running averages from training, so a prediction doesn't depend on the other examples (and a batch of one works).
</details>

<details><summary>5. When would you use layer norm or group norm instead of batch norm?</summary>

When batch statistics are unreliable or unavailable: small or variable batches (detection, segmentation: group norm), sequence models and Transformers (layer norm), or when train/test consistency matters.
</details>

### Exercises
1. Add **dilation** to `im2col` (step by `dilation` between filter taps) and check the effective kernel size $d(K - 1) + 1$ with `measured_receptive_field`.
2. Add a **second conv layer** (8 → 16 filters) to `SmallCNN`. How do accuracy and shift robustness change?
3. Implement **spatial batch norm** (statistics over N, H, W) by reshaping to $(N\cdot H\cdot W)\times C$ and reusing `BatchNorm`; add it to `SmallCNN`.
4. Gradient-check **average pooling**'s backward pass.
5. Use §4.2's backprop trick on the trained `SmallCNN`: which input pixels most affect the correct class's score? (A first "saliency map", Lecture 14.)

### References
* CS231n course notes, *Convolutional Neural Networks* (cs231n.github.io/convolutional-networks).
* LeCun et al. 1998; Krizhevsky et al. 2012; Lin, Chen & Yan 2014 (Network in Network); Springenberg et al. 2015.
* Yu & Koltun 2016 (dilated convolutions); Zhang 2019 (shift invariance); Chellapilla et al. 2006 (im2col); Chetlur et al. 2014 (cuDNN).
* Ioffe & Szegedy 2015 (batch norm); Santurkar et al. 2018; Ba, Kiros & Hinton 2016 (layer norm); Ulyanov et al. 2016 (instance norm); Wu & He 2018 (group norm).

**Next:** Lecture 8, CNN Architectures.